# TFT v3 — Groundwater Forecasting, t+7 / t+14 / t+30 at q10 / q50 / q90

Rebuilt from `09_tft_colab_production_v2.ipynb` after a full audit of the v2 run and the
data pipeline behind it. Every change below is motivated by a measured number, not a guess.
The audit and its evidence are written up in **`V3_PLAN.md`**.

---

### The finding that shaped this notebook

An independent **predictability probe** — ridge regression on the same Δ-target, per district,
same walk-forward folds — showed that **6 of the 7 districts where v2 loses to persistence also
lose with ridge**: Aurangabad, Bhandara, Nashik, Jalgaon, Dhule, Kolhapur. Two unrelated model
families failing on exactly the same districts is a **data limitation, not a TFT bug**.
`PROD_V1_vs_V2.md` §7 left this "open rather than guessed at" — this is the answer.

What actually drives district predictability (Spearman vs. ridge MASE @ t+30):

| Driver | ρ |
|---|---|
| 2023 observed rate | **−0.54** |
| overall observed rate | **−0.54** |
| number of stations | **−0.51** |
| size of 30-day movement | −0.10 (nothing) |

**More wells and a longer complete record make a district forecastable. Movement size does not.**
Districts are therefore assigned a **data tier** in `merge_master.py`, from coverage and station
count only — never fitted to model results — so the tier can be quoted as an independent
explanation of the per-district table.

---

### What changed from v2

**Data pipeline (upstream, already applied)**

| # | Problem | Fix |
|---|---|---|
| D1 | ET & soil moisture were **100% day-of-year climatology after 2025-06-30** — 0% real in folds 5–6, 25% in fold 4 | Swapped to **ERA5-Land** (`Data Collection/era5_land.py`), ~5-day latency. Also adds 4 soil-moisture depths — `sm_l3` (28–100 cm) and `sm_l4` (100–289 cm) sit on the actual recharge path — plus PET and a water-stress term |
| D2 | Sentinel monthly composites stamped at the **1st** of the month then linearly interpolated → up to **15 days of future** in every daily NDVI value | Stamped at month-**end**, held constant (`ffill`). Strictly causal |
| D3 | Districts missing most of 2023 got **20 training windows vs 49**, and the inverse-frequency sampler then **oversampled them 2.45×** | Sampler cap 3.0 → **1.5** |
| D4 | **Latur** — 44% observed, 1 station, **0 test windows in folds 4–6** | Tier D, excluded from training |
| D5 | Median-fills computed over **all** rows including the test period | All fills moved inside `normalize_for_fold()`, train rows only |

**Model / evaluation**

| # | Problem | Fix |
|---|---|---|
| M1 | Nothing stopped a district being far worse than persistence | **Shrinkage toward persistence**, λ(district, horizon-band) fitted on a held-out calibration block. λ=0 *is* persistence, so no district can blow up |
| M2 | One `delta_sd` pooled over h=1…30 ⇒ the loss was dominated by long horizons and **the model had almost no gradient incentive at t+7** | **Per-horizon target scaling** `delta_sd[district][h]` |
| M3 | Validation was **9 windows per district**; early stopping was a coin flip (best epoch swung 3→17) | `VAL_MONTHS` 2→3, validation stride 7→2, and the span is **split**: early stopping on the first 60%, λ + conformal on the last 40% |
| M4 | Conformal margins were computed **on the test set they calibrate** — "80% coverage" was a mathematical identity | Margins fitted on the **calibration block**, applied to test. The reported number is now a real out-of-sample coverage |
| M5 | q10/q50/q90 were independent outputs and could cross | **Monotone head**: q50 ± softplus spreads |
| M6 | Single seed ⇒ per-district results were partly luck | **3-seed ensemble** |
| M7 | Only baseline was persistence | Full ladder: persistence · climatology-of-change · **ridge** · **LightGBM** · TFT |
| M8 | Rainfall entered only as 7/30-day sums | **API** (exponential-decay antecedent precipitation), **cumulative rainfall departure**, 60/90-day sums, days-since-rain, dry-spell length |
| M10 | MAPE reported without its caveat | MAPE **and** MASE **and** skill score **and** NSE **and** pinball loss, plus **Diebold–Mariano** significance and bootstrap CIs |
| M11 | Only the daily district median was scored | Daily **and** 7-day-mean targets both reported |
| M14 | v1→v2 was not comparable (data layer changed underneath) | **Regression gate** — per-district v2→v3 delta table, degradations >10% flagged |
| — | v2's notebook reported **205,173 parameters**; that config actually builds **1,231,159** | Capacity is now genuinely controllable (`D_SELECT`, `GRN_HIDDEN_FACTOR`) and the true count is printed with a params-per-independent-window ratio |

---

### How to read the results

**MASE < 1** is the criterion of success — the model beats "assume nothing changes".
**Skill score** = `1 − MSE_model/MSE_persistence`, i.e. *"reduces squared error by X% vs assuming
no change"* — the same information in the form that is easiest to defend out loud.

> ⚠️ **MAPE caveat, state this whenever you show it.** Groundwater depth is measured from an
> arbitrary datum, so MAPE rewards deep districts. A 5% error is **2.1 m in Dhule** (mean level
> −42.9 m) and **0.16 m in Raigad** (−3.25 m). MAPE will rank districts almost *inversely* to real
> skill. Always show it beside MAE and MASE.


In [ ]:
# ── STEP 1: Upload files ─────────────────────────────────────────────────────
# Required : tft_model_v3.py, master_dataset.csv  (the REBUILT one — 55 cols, has `data_tier`)
# Optional : tft_metrics_per_district_v2.csv      (enables the v2 -> v3 regression gate)
from google.colab import files
print('Upload tft_model_v3.py + master_dataset.csv (+ optionally the v2 per-district metrics)')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install -q lightgbm scikit-learn
print('Done')

In [ ]:
import sys, warnings, json, time, math, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import torch BEFORE sklearn — sklearn-then-torch can segfault in torch.optim on
# some platforms (BLAS/OpenMP DLL conflict). Harmless on Colab, kept for portability.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False
    from sklearn.ensemble import HistGradientBoostingRegressor

from tft_model_v3 import TemporalFusionTransformer, MaskedQuantileLoss

warnings.filterwarnings('ignore')

ALLOW_CPU = False   # True only for a deliberate CPU smoke test with a shrunk config

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise RuntimeError(
        "No GPU detected. Runtime -> Change runtime type -> T4 GPU -> Save, then re-run every "
        "cell from the top (a GPU switch gives a fresh VM, so re-upload the files).")
print('LightGBM available:', HAS_LGB)

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
ENCODER_LEN   = 60
DECODER_LEN   = 30
EVAL_STEPS    = [6, 13, 29]                      # 0-indexed -> t+7, t+14, t+30
HNAME         = {6: 't+7', 13: 't+14', 29: 't+30'}
QUANTILES     = [0.1, 0.5, 0.9]

# ── Capacity ────────────────────────────────────────────────────────────────
# v2's notebook printed "Model parameters: 205,173". That is wrong: the same
# config actually instantiates 1,231,159 parameters. The dominant block is the
# variable-selection GRN, whose first layer is Linear(n_past * d_model, d_model)
# — 143k parameters by itself, plus a 78k skip. So the *variable selector* was
# bigger than the rest of the network, on a problem with ~290 genuinely
# independent training windows.
#
# tft_model_v3 compresses each variable to D_SELECT dims before the selector
# sees it, and narrows the per-variable GRN bottleneck by GRN_HIDDEN_FACTOR.
# The true count and the params-per-independent-window ratio are printed below —
# watch them, they are the overfitting budget.
D_MODEL           = 32
D_SELECT          = 8
GRN_HIDDEN_FACTOR = 0.5
N_HEADS           = 4
LSTM_LAYERS       = 1
DROPOUT           = 0.25
CAT_EMBED_DIM     = 8
WEIGHT_DECAY      = 1e-2

# ── Optimisation ────────────────────────────────────────────────────────────
BATCH_SIZE    = 128
EVAL_BATCH    = 512
MAX_EPOCHS    = 60
LR            = 1e-3
WARMUP_EPOCHS = 5
PATIENCE      = 10
GRAD_CLIP     = 1.0
LR_PATIENCE, LR_FACTOR, MIN_LR = 4, 0.5, 1e-5

# ── Seed ensemble (M6) ──────────────────────────────────────────────────────
# Averaging 3 seeds removes the "district X happened to fail this run" variance
# that made v2's per-district table hard to interpret. Set SEEDS=[42] for a
# single-seed smoke run.
SEEDS = [42, 1337, 2026]

# ── Windows ─────────────────────────────────────────────────────────────────
# Consecutive stride-1 windows share ENCODER_LEN+DECODER_LEN-1 days, so they are
# near-duplicates. Training steps a week at a time. Validation uses stride 2
# (v2 used 7, which gave only ~9 val windows per district — early stopping was
# a coin flip). Test stays stride 1 for full forecast-origin coverage.
TRAIN_STRIDE, VAL_STRIDE, TEST_STRIDE = 7, 2, 1
ENC_OBS_MIN, DEC_OBS_MIN = 0.70, 0.50

# Inverse per-district window-count weighting. v2 capped at 3.0, which meant the
# districts with the FEWEST windows (Beed/Aurangabad: 20 in fold1, vs Nagpur's
# 49) were sampled 2.45x more often — the model saw the same 20 windows over and
# over, in exactly the districts with the least real history. Capped tighter here.
SAMPLE_WEIGHT_CAP = 1.5

# ── Validation split (M3) ───────────────────────────────────────────────────
# The validation span is split in two. Early stopping may only see the first
# part; lambda-shrinkage and conformal margins are fitted on the second part,
# which no training decision has touched. This is what makes the calibrated
# coverage a real out-of-sample number instead of v2's self-fulfilling one.
VAL_MONTHS  = 3
VAL_ES_FRAC = 0.60

# ── Calibration (M1 / M4) ───────────────────────────────────────────────────
TARGET_COVERAGE = 0.80
LAMBDA_GRID     = np.round(np.arange(0.0, 1.0001, 0.05), 3)
LAMBDA_PRIOR_N  = 200     # hierarchical strength: pulls a district's lambda toward the global one
CONF_MIN_N      = 50      # below this, pool conformal scores across districts
BAND_SIZE       = 10      # horizon bands for lambda / conformal: steps 0-9, 10-19, 20-29

# ── District gating ─────────────────────────────────────────────────────────
EXCLUDE_TIERS = ['D']     # Latur: 44% observed, 1 station, 0 test windows in folds 4-6
HEADLINE_EXCLUDE_LOW_STATION = True

FOLDS = [
    {"name": "fold1_monsoon2024",     "test_start": "2024-07-01", "test_end": "2024-09-30"},
    {"name": "fold2_postmonsoon2024", "test_start": "2024-10-01", "test_end": "2024-12-31"},
    {"name": "fold3_dry2025",         "test_start": "2025-01-01", "test_end": "2025-05-31"},
    {"name": "fold4_monsoon2025",     "test_start": "2025-06-01", "test_end": "2025-09-30"},
    {"name": "fold5_postmonsoon2025", "test_start": "2025-10-01", "test_end": "2025-12-31"},
    {"name": "fold6_dry2026",         "test_start": "2026-01-01", "test_end": "2026-06-04"},
]

OUT_DIR   = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('master_dataset.csv')
V2_METRICS = Path('tft_metrics_per_district_v2.csv')   # optional, for the regression gate

print(f'Capacity : d_model={D_MODEL}, d_select={D_SELECT}, grn_hidden={GRN_HIDDEN_FACTOR}, '
      f'layers={LSTM_LAYERS}, dropout={DROPOUT}, wd={WEIGHT_DECAY}')
print(f'Windows  : enc={ENCODER_LEN}, dec={DECODER_LEN}, '
      f'stride train/val/test = {TRAIN_STRIDE}/{VAL_STRIDE}/{TEST_STRIDE}')
print(f'Seeds    : {SEEDS}   ({len(SEEDS)} x {len(FOLDS)} = {len(SEEDS)*len(FOLDS)} training runs)')
print(f'Folds    : {[f["name"] for f in FOLDS]}')

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values(['district','date']).reset_index(drop=True)
print(f'Shape: {df.shape} | Districts: {df["district"].nunique()} | '
      f'{df["date"].min().date()} -> {df["date"].max().date()}')

SOIL_COLS     = ['sand_0cm','clay_0cm','silt_0cm','sand_10cm','clay_10cm','silt_10cm',
                 'sand_30cm','clay_30cm','silt_30cm']
SENTINEL_COLS = ['ndvi_mean','ndwi_mean','ndmi_mean','vv_db','vh_db','vh_vv_ratio']
CGWB_COLS     = ['annual_recharge_bcm','annual_extraction_bcm',
                 'extractable_resource_bcm','stage_of_extraction_pct']

# ERA5-Land extras — present only after era5_land.py has been run and the
# pipeline re-built. Everything below degrades gracefully without them.
ERA5_COLS = [c for c in ['pet_mm','water_stress','sm_l1','sm_l2','sm_l3','sm_l4'] if c in df.columns]
HAS_ERA5  = len(ERA5_COLS) > 0
print(f'ERA5-Land columns present: {ERA5_COLS if HAS_ERA5 else "NONE (running on the old WRIS feed)"}')

df['aquifer_category_id'] = df['aquifer_category_id'].fillna(0).astype(int)
df['season_id'] = df['season'].map({'winter':0,'summer':1,'monsoon':2,'post_monsoon':3}).astype(int)

# data_tier is produced by merge_master.py; recompute if running against an older master
if 'data_tier' not in df.columns:
    print('WARNING: no `data_tier` column — recomputing inline (re-run merge_master.py to persist it)')
    c23 = df[df.date.dt.year == 2023].groupby('district')['gwl_observed'].mean()
    call = df.groupby('district')['gwl_observed'].mean()
    nst  = df.groupby('district')['n_stations'].first()
    def _t(d):
        if call[d] < 0.55: return 'D'
        if nst[d] >= 6 and c23.get(d,0) >= 0.65: return 'A'
        if nst[d] <= 4 and c23.get(d,0) <  0.55: return 'C'
        return 'B'
    df['data_tier'] = df['district'].map({d:_t(d) for d in df.district.unique()})

# ── RAW target vs INTERPOLATED encoder series ───────────────────────────────
# gwl_mean_raw keeps true NaN on unobserved days and is used ONLY for loss/eval.
# gwl_mean is interpolated and used ONLY as a model input. All GWL stays in
# METRES here — the Dataset applies the anchor-relative scaling per window,
# because the anchor differs from window to window.
df['gwl_mean_raw'] = df['gwl_mean'].copy()
for col in ['gwl_mean','gwl_min','gwl_max']:
    df[col] = df.groupby('district')[col].transform(lambda s: s.interpolate('linear', limit_direction='both'))

df['gwl_lag_7']  = df.groupby('district')['gwl_mean'].shift(7)
df['gwl_lag_14'] = df.groupby('district')['gwl_mean'].shift(14)
df['gwl_lag_30'] = df.groupby('district')['gwl_mean'].shift(30)
df['gwl_7d_change']  = df['gwl_mean'] - df['gwl_lag_7']
df['gwl_30d_change'] = df['gwl_mean'] - df['gwl_lag_30']

print(f'gwl_observed rate: {df["gwl_observed"].mean():.1%}')

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
# All strictly past-only or calendar-derived. Nothing here can see the future.

df['month_sin'] = np.sin(2*np.pi*df['month']/12.0);        df['month_cos'] = np.cos(2*np.pi*df['month']/12.0)
df['doy_sin']   = np.sin(2*np.pi*df['day_of_year']/365.25); df['doy_cos']  = np.cos(2*np.pi*df['day_of_year']/365.25)

for c in ['rainfall_mm','rainfall_7d','rainfall_30d']:
    df[c+'_log'] = np.log1p(df[c].clip(lower=0))

# ── M8: hydrologically-motivated rainfall features ──────────────────────────
# Rainfall -> recharge lags 3-21 days in these aquifers, so a fixed 7- or 30-day
# sum is a blunt instrument. These are the standard alternatives:
#
#   API  (Antecedent Precipitation Index) — exponentially-weighted memory of past
#        rain. k=0.90 -> ~7-day memory, 0.95 -> ~14-day, 0.98 -> ~35-day. Lets the
#        model pick its own recharge lag instead of being handed one.
#   CRD  (Cumulative Rainfall Departure) — running sum of (rain - long-run mean).
#        The classic predictor of water-table position in hard-rock aquifers:
#        it tracks the accumulated wet/dry balance rather than any single event.
#   dry-spell length / days since rain — recession behaviour between events.
def _api(s, k):
    out = np.zeros(len(s), dtype=np.float64)
    acc = 0.0
    for i, v in enumerate(s):
        acc = acc * k + (v if np.isfinite(v) else 0.0)
        out[i] = acc
    return out

parts = []
for dist, g in df.groupby('district', sort=False):
    g = g.sort_values('date').copy()
    r = g['rainfall_mm'].fillna(0.0).values
    for k, name in [(0.90,'api_fast'), (0.95,'api_mid'), (0.98,'api_slow')]:
        g[name] = _api(r, k)
    g['rainfall_60d'] = g['rainfall_mm'].rolling(60, min_periods=1).sum()
    g['rainfall_90d'] = g['rainfall_mm'].rolling(90, min_periods=1).sum()
    # CRD against this district's own daily mean rainfall
    g['crd'] = (g['rainfall_mm'].fillna(0.0) - g['rainfall_mm'].mean()).cumsum()
    wet = (g['rainfall_mm'].fillna(0.0) > 1.0).values
    dsr, run = np.zeros(len(g)), 0
    for i, w in enumerate(wet):
        run = 0 if w else run + 1
        dsr[i] = run
    g['days_since_rain'] = np.minimum(dsr, 120)
    parts.append(g)
df = pd.concat(parts, ignore_index=False).sort_index()

RAIN_EXTRA = ['api_fast','api_mid','api_slow','rainfall_60d','rainfall_90d','crd','days_since_rain']
for c in ['rainfall_60d','rainfall_90d']:
    df[c] = np.log1p(df[c].clip(lower=0))

# ── Rolling GWL statistics (shift(1) => strictly past) ──────────────────────
for w in [30, 60, 90]:
    df[f'gwl_roll_mean_{w}'] = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).mean())
    df[f'gwl_roll_std_{w}']  = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).std())
ROLL_COLS = [c for c in df.columns if c.startswith('gwl_roll_')]

# NOTE: no median-filling here. v2 filled these with the per-district median over
# the FULL series, including the test period (D5). All fills now happen inside
# normalize_for_fold(), computed on training rows only.
print(f'Engineered: 4 cyclic + 3 log-rainfall + {len(RAIN_EXTRA)} hydro + {len(ROLL_COLS)} rolling. '
      f'Total cols: {df.shape[1]}')

In [ ]:
# ── Feature definitions ──────────────────────────────────────────────────────
STATIC_CAT  = ['district_id', 'aquifer_category_id']
STATIC_CONT = SOIL_COLS + CGWB_COLS

# Column roles for normalisation — the core of the delta reframing:
#   LEVEL_COLS  absolute depths  -> (value - anchor) / sd_enc, per window
#   DIFF_COLS   already changes  -> value / sd_enc
#   FLAG_COLS   0/1 indicators   -> untouched
#   everything else              -> ordinary train-fitted StandardScaler
LEVEL_COLS = ['gwl_mean','gwl_min','gwl_max','gwl_lag_7','gwl_lag_14','gwl_lag_30',
              'gwl_roll_mean_30','gwl_roll_mean_60','gwl_roll_mean_90']
DIFF_COLS  = ['gwl_7d_change','gwl_30d_change','gwl_roll_std_30','gwl_roll_std_60','gwl_roll_std_90']
FLAG_COLS  = [c for c in ['gwl_observed','et_mm_observed','soil_moisture_observed'] if c in df.columns]

PAST_CONT = (LEVEL_COLS + DIFF_COLS + FLAG_COLS + [
    'station_count',
    'rainfall_mm_log','rainfall_7d_log','rainfall_30d_log',
    'et_mm','soil_moisture','net_water_balance',
    'temp_max','temp_min','temp_mean','humidity_mean','vpd_approx',
] + RAIN_EXTRA + ERA5_COLS + SENTINEL_COLS)

# Known-future covariates. Calendar only + an explicit horizon index: with a
# direct multi-horizon head the model otherwise has to infer "how far ahead am I"
# from position alone.
FUTURE_CONT = ['month_sin','month_cos','doy_sin','doy_cos','is_monsoon','season_id','horizon_frac']

TARGET, LEVEL, OBS_COL = 'gwl_mean_raw', 'gwl_mean', 'gwl_observed'

# horizon_frac is written per decoder step by the Dataset, but needs to exist as a
# column so the scaler sees a consistent schema. It is deliberately NOT scaled.
df['horizon_frac'] = 0.0

N_DISTRICTS  = int(df['district_id'].max()) + 1
N_CATEGORIES = int(df['aquifer_category_id'].max()) + 1

TIER = df.drop_duplicates('district').set_index('district')['data_tier'].to_dict()
LOW_STATION = df.drop_duplicates('district').set_index('district')['low_station_flag'].to_dict()
DROP_DISTRICTS = sorted([d for d, t in TIER.items() if t in EXCLUDE_TIERS])

print(f'Past={len(PAST_CONT)}  Future={len(FUTURE_CONT)}  Static={len(STATIC_CONT)}+{len(STATIC_CAT)}cat')
print(f'Districts={df["district"].nunique()}, aquifer categories={N_CATEGORIES}')
print(f'Excluded (tier {EXCLUDE_TIERS}): {DROP_DISTRICTS}')
print(f'Low-station (reported separately): {sorted([d for d,f in LOW_STATION.items() if f==1])}')

df = df[~df['district'].isin(DROP_DISTRICTS)].reset_index(drop=True)
print(f'Modelling {df["district"].nunique()} districts.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# DATA-QUALITY REPORT — read this BEFORE the model results.
# It states what the data can and cannot support, so the per-district table
# later can be interpreted rather than just ranked.
# ══════════════════════════════════════════════════════════════════════════
print('='*86)
print('1. COVARIATE REALITY CHECK — fraction of each test fold with GENUINE (not climatology) data')
print('='*86)
obs_flags = [c for c in ['et_mm_observed','soil_moisture_observed','pet_mm_observed'] if c in df.columns]
rows = []
for f in FOLDS:
    m = (df.date >= f['test_start']) & (df.date <= f['test_end'])
    r = {'fold': f['name'], 'gwl': round(df.loc[m,'gwl_observed'].mean(), 2)}
    for c in obs_flags:
        r[c.replace('_observed','')] = round(df.loc[m, c].mean(), 2)
    rows.append(r)
cov_report = pd.DataFrame(rows).set_index('fold')
print(cov_report.to_string())
bad = {c: cov_report[c].min() for c in cov_report.columns if cov_report[c].min() < 0.5}
if bad:
    print(f'\n  *** WARNING: {list(bad)} are climatology-filled in at least one fold '
          f'(min coverage {min(bad.values()):.0%}).')
    print('      Those features carry no information beyond doy_sin/doy_cos there.')
    print('      Fix: cd "Data Collection" && python era5_land.py, then re-run the preprocessing.')
else:
    print('\n  OK — every covariate is real data in every fold.')

print('\n' + '='*86)
print('2. DATA TIER — assigned from station count and record completeness ONLY,')
print('   never from model results, so it is an independent explanation of them.')
print('='*86)
q = df.groupby('district').agg(
    tier=('data_tier','first'), n_st=('n_stations','first'),
    obs_all=('gwl_observed','mean'),
).round(3)
q['obs_2023'] = df[df.date.dt.year==2023].groupby('district')['gwl_observed'].mean().round(3)
# the quantity the model has to predict: typical real movement over each horizon
sig = {}
for d, g in df.groupby('district'):
    s = pd.Series(np.where(g['gwl_observed'].values==1, g['gwl_mean'].values, np.nan))
    sig[d] = {f'move_{h}d': round(float((s - s.shift(h)).abs().median()), 3) for h in (7,14,30)}
q = q.join(pd.DataFrame(sig).T)
q = q.sort_values(['tier','obs_2023'], ascending=[True, False])
print(q.to_string())
for t in ['A','B','C']:
    mem = sorted(q.index[q.tier==t])
    print(f'  Tier {t} ({len(mem):2d}): {mem}')
print('''
  Tier A  dense panel (>=6 wells) + >=65% of 2023 observed  -> headline results
  Tier B  usable
  Tier C  <=4 wells AND <55% of 2023 observed -> a DATA finding, not a model failure.
          In the ridge probe, 6 of these lose to persistence for a linear model too.
  `move_7d` is the median real 7-day movement. Where that is ~0.1 m, no model can
  beat persistence at t+7 — the movement is smaller than the measurement noise.''')
q.to_csv(OUT_DIR/'data_quality_scorecard.csv')

In [ ]:
# ── Per-fold normalisation, train-only (D5 + M2) ─────────────────────────────
SCALED_PAST = [c for c in PAST_CONT if c not in LEVEL_COLS + DIFF_COLS + FLAG_COLS]
SCALED_FUT  = [c for c in FUTURE_CONT if c not in ('horizon_frac',)]
FILL_COLS   = SOIL_COLS + SENTINEL_COLS + CGWB_COLS + ROLL_COLS + ERA5_COLS


def compute_delta_scales(d, fit_end, decoder_len=DECODER_LEN):
    '''Per-district scale of what the model actually predicts.

    M2: v2 pooled the h-step change scale over h=1..30 into ONE number per
    district. In scaled units a 7-day change was then roughly half the size of a
    30-day change, and the pinball loss — averaged uniformly over the 30 decoder
    steps — was dominated by the long horizons. The model had almost no gradient
    incentive at t+7, which is a mechanical reason (on top of the physical one)
    for v2's t+7 MASE of 1.004.

    Now every horizon gets its own scale, so each decoder step contributes
    comparably to the loss. Reconstruction stays  level = anchor + out[h]*sd[h],
    so an output of 0 is still exactly persistence.

    sd_enc (used for the encoder's level/diff features, where there is no single
    horizon) is the RMS of the per-horizon scales.
    '''
    tr = d[d['date'] <= pd.Timestamp(fit_end)]
    sd_h, sd_enc = {}, {}
    for dist, g in tr.groupby('district'):
        s = g.sort_values('date')[LEVEL]
        per_h = []
        for h in range(1, decoder_len + 1):
            v = (s - s.shift(h)).dropna().values
            per_h.append(float(np.std(v)) if len(v) > 10 else np.nan)
        per_h = pd.Series(per_h).interpolate().bfill().ffill().values
        per_h = np.maximum(np.nan_to_num(per_h, nan=1.0), 1e-3)
        sd_h[dist]   = per_h.astype(np.float64)
        sd_enc[dist] = float(np.sqrt(np.mean(per_h ** 2)))
    return sd_enc, sd_h


def normalize_for_fold(d, fit_end):
    '''Fit every fill and every scaler on rows <= fit_end ONLY.

    v2 computed the soil/Sentinel/CGWB medians and the rolling-warmup fills over
    the whole frame, test period included (D5). Small in effect, but it is the
    first thing a reviewer checks.
    '''
    fit_end = pd.Timestamp(fit_end)
    tr = d['date'] <= fit_end
    dn = d.copy()

    for c in FILL_COLS:
        if c not in dn.columns:
            continue
        med_by_dist = dn.loc[tr].groupby('district')[c].median()
        dn[c] = dn[c].fillna(dn['district'].map(med_by_dist))
        dn[c] = dn[c].fillna(dn.loc[tr, c].median()).fillna(0.0)

    feat = [c for c in SCALED_PAST + STATIC_CONT + SCALED_FUT if c in dn.columns]
    scaler = StandardScaler().fit(dn.loc[tr, feat])
    dn[feat] = scaler.transform(dn[feat])

    sd_enc, sd_h = compute_delta_scales(d, fit_end)
    return dn, sd_enc, sd_h


print('normalize_for_fold() + compute_delta_scales() defined (train-only, per-horizon).')

In [ ]:
# ── Dataset: calendar-true windows, anchor-relative, per-horizon delta target ─
class GWLDataset(Dataset):
    '''Every window is ENCODER_LEN + DECODER_LEN consecutive days.

    Everything is expressed RELATIVE TO THE ANCHOR (the last encoder day's
    interpolated level):
        anchor          = level[i-1]
        level feature  -> (value - anchor) / sd_enc
        diff feature   -> value / sd_enc
        target step h  -> (raw_level[i+h] - anchor) / sd_h[h]        <-- M2
    so the network never sees an absolute depth and the problem is
    translation-invariant. Reconstruction is level = anchor + out[h]*sd_h[h],
    which means an output of exactly 0 reproduces persistence.

    The anchor MUST be a real reading. `level` is the interpolated series; if day
    i-1 were unobserved its value would be derived from the next observed point,
    which can lie inside the decoder window — leaking the future into the anchor.
    Requiring obs[i-1]==1 also guarantees every interior encoder gap is bounded by
    observed points inside the encoder.
    '''

    def __init__(self, d, past_cols, future_cols, static_cat_cols, static_cont_cols,
                 sd_enc, sd_h, split_start=None, split_end=None, stride=1,
                 encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN):
        self.encoder_len, self.decoder_len = encoder_len, decoder_len
        lvl_idx  = [past_cols.index(c) for c in LEVEL_COLS if c in past_cols]
        diff_idx = [past_cols.index(c) for c in DIFF_COLS  if c in past_cols]
        hf_idx   = future_cols.index('horizon_frac')
        horizon_frac = ((np.arange(decoder_len) + 1) / decoder_len).astype(np.float32)

        P, F, T, M = [], [], [], []
        SC, SX, TAB = [], [], []
        self.district, self.date, self.anchor_date = [], [], []
        self.anchor, self.sd_mat, self.actual, self.did = [], [], [], []
        self.n_drop_enc = self.n_drop_dec = self.n_drop_anchor = 0

        for dist, grp in d.groupby('district'):
            grp = grp.sort_values('date').reset_index(drop=True)
            n = len(grp)
            obs  = grp[OBS_COL].values
            lvl  = grp[LEVEL].values.astype(np.float64)
            past = grp[past_cols].values.astype(np.float64)
            fut  = grp[future_cols].values.astype(np.float32)
            tgt  = grp[TARGET].values.astype(np.float64)
            dates = grp['date'].values
            sc = grp.loc[0, static_cat_cols].values.astype(np.int64)
            sx = grp.loc[0, static_cont_cols].values.astype(np.float32)
            se = float(sd_enc.get(dist, 1.0))
            sh = sd_h.get(dist, np.ones(decoder_len))
            did = int(grp.loc[0, 'district_id'])

            for i in range(encoder_len, n - decoder_len + 1, stride):
                te = dates[i + decoder_len - 1]
                if split_start is not None and te < np.datetime64(pd.Timestamp(split_start)): continue
                if split_end   is not None and te > np.datetime64(pd.Timestamp(split_end)):   continue
                if obs[i-encoder_len:i].mean() < ENC_OBS_MIN: self.n_drop_enc += 1;    continue
                if obs[i:i+decoder_len].mean() < DEC_OBS_MIN: self.n_drop_dec += 1;    continue
                if obs[i-1] != 1:                             self.n_drop_anchor += 1; continue

                a = lvl[i-1]
                if not np.isfinite(a): continue

                pc = past[i-encoder_len:i].copy()
                if lvl_idx:  pc[:, lvl_idx]  = (pc[:, lvl_idx] - a) / se
                if diff_idx: pc[:, diff_idx] = pc[:, diff_idx] / se
                fc = fut[i:i+decoder_len].copy()
                fc[:, hf_idx] = horizon_frac

                raw = tgt[i:i+decoder_len]
                tg  = (raw - a) / sh
                mk  = obs[i:i+decoder_len].astype(np.float32)

                if not np.isfinite(pc).all() or not np.isfinite(fc).all(): continue

                P.append(pc.astype(np.float32)); F.append(fc)
                T.append(np.nan_to_num(tg, nan=0.0).astype(np.float32)); M.append(mk)
                SC.append(sc); SX.append(sx)
                # anchor-day snapshot at three time scales — the feature table the
                # ridge / LightGBM baselines see, so they get exactly the same
                # windows and the same information the TFT encoder has.
                TAB.append(np.concatenate([pc[-1], pc[-7:].mean(0), pc[-30:].mean(0)]).astype(np.float32))
                self.district.append(dist); self.date.append(pd.Timestamp(te))
                self.anchor_date.append(pd.Timestamp(dates[i-1]))
                self.anchor.append(float(a)); self.sd_mat.append(sh.astype(np.float32))
                self.actual.append(raw.astype(np.float64)); self.did.append(did)

        self.P  = np.asarray(P,  np.float32) if P else np.zeros((0, encoder_len, len(past_cols)), np.float32)
        self.F  = np.asarray(F,  np.float32) if F else np.zeros((0, decoder_len, len(future_cols)), np.float32)
        self.T  = np.asarray(T,  np.float32) if T else np.zeros((0, decoder_len), np.float32)
        self.M  = np.asarray(M,  np.float32) if M else np.zeros((0, decoder_len), np.float32)
        self.SC = np.asarray(SC, np.int64)   if SC else np.zeros((0, len(static_cat_cols)), np.int64)
        self.SX = np.asarray(SX, np.float32) if SX else np.zeros((0, len(static_cont_cols)), np.float32)
        self.tab     = np.asarray(TAB, np.float32) if TAB else np.zeros((0, 3*len(past_cols)), np.float32)
        self.anchor  = np.asarray(self.anchor,  np.float64)
        self.sd_mat  = np.asarray(self.sd_mat,  np.float32) if len(self.sd_mat) else np.zeros((0, decoder_len), np.float32)
        self.actual  = np.asarray(self.actual,  np.float64) if len(self.actual) else np.zeros((0, decoder_len))
        self.did     = np.asarray(self.did,     np.int64)
        self.district = np.asarray(self.district)
        self.date     = pd.to_datetime(pd.Series(self.date)) if len(self.date) else pd.Series([], dtype='datetime64[ns]')
        self.anchor_date = pd.to_datetime(pd.Series(self.anchor_date)) if len(self.anchor_date) else pd.Series([], dtype='datetime64[ns]')

    def __len__(self): return len(self.P)

    def __getitem__(self, i):
        return (torch.from_numpy(self.SC[i]), torch.from_numpy(self.SX[i]),
                torch.from_numpy(self.P[i]),  torch.from_numpy(self.F[i]),
                torch.from_numpy(self.T[i]),  torch.from_numpy(self.M[i]),
                i)

    def counts(self):
        return Counter(self.district.tolist())


def build_sampler(ds, cap=SAMPLE_WEIGHT_CAP):
    '''Inverse per-district window-count weights, capped.

    D3: v2 capped at 3.0, so Beed/Aurangabad (20 windows in fold1) were sampled
    2.45x more often than Nagpur (49) — the model saw the same handful of windows
    repeatedly in exactly the districts with the least real history. A tighter cap
    still corrects volume imbalance without amplifying the scarcity.
    '''
    counts = ds.counts()
    raw = {d: 1.0/c for d, c in counts.items()}
    mn = min(raw.values())
    capped = {d: min(w, mn*cap) for d, w in raw.items()}
    return WeightedRandomSampler([capped[d] for d in ds.district], num_samples=len(ds), replacement=True)


print('GWLDataset (per-horizon delta target + baseline feature table) and build_sampler defined.')

In [ ]:
# ── Model factory + training loop ────────────────────────────────────────────
def build_model(seed=None):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    return TemporalFusionTransformer(
        n_static_cat=len(STATIC_CAT), cat_vocab_sizes=[N_DISTRICTS, N_CATEGORIES],
        cat_embed_dim=CAT_EMBED_DIM, n_static_cont=len(STATIC_CONT),
        n_past=len(PAST_CONT), n_future=len(FUTURE_CONT),
        encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN,
        d_model=D_MODEL, n_heads=N_HEADS, lstm_layers=LSTM_LAYERS,
        dropout=DROPOUT, quantiles=QUANTILES,
        monotone_quantiles=True, d_select=D_SELECT, grn_hidden_factor=GRN_HIDDEN_FACTOR,
    ).to(DEVICE)


def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for sc, sx, px, fx, y, mk, _ in loader:
            sc, sx, px, fx, y, mk = [t.to(DEVICE, non_blocking=True) for t in (sc, sx, px, fx, y, mk)]
            preds, _, _ = model(sc, sx, px, fx)
            loss = criterion(preds, y, mk)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            total += loss.item(); n += 1
    return total / max(n, 1)


def train_one(tag, train_ds, es_ds, seed, verbose=True):
    '''Train one (fold, seed). Early stopping sees ONLY the es_ds block —
    the calibration block is kept clean for lambda and conformal (M3/M4).'''
    model = build_model(seed)
    criterion = MaskedQuantileLoss(QUANTILES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR)

    # num_workers=0 — the dataset is fully materialised in RAM, so workers add no
    # I/O parallelism but fork a full copy each (this crashed Colab in v1).
    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=build_sampler(train_ds),
                    num_workers=0, pin_memory=(DEVICE.type=='cuda'), drop_last=False)
    vl = DataLoader(es_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=0,
                    pin_memory=(DEVICE.type=='cuda'))

    tr_hist, va_hist = [], []
    best, wait, best_state = float('inf'), 0, None
    for ep in range(1, MAX_EPOCHS+1):
        if ep <= WARMUP_EPOCHS:
            for g in optimizer.param_groups: g['lr'] = LR * ep / WARMUP_EPOCHS
        tr = run_epoch(tl, model, criterion, optimizer)
        va = run_epoch(vl, model, criterion)
        tr_hist.append(tr); va_hist.append(va)
        if ep > WARMUP_EPOCHS: plateau.step(va)
        if va < best:
            best, wait = va, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
        if verbose and (ep <= 3 or ep % 5 == 0):
            print(f'    [{tag}] ep {ep:03d} train={tr:.4f} val={va:.4f} '
                  f'lr={optimizer.param_groups[0]["lr"]:.1e}')
    model.load_state_dict(best_state)
    if verbose:
        print(f'    [{tag}] stopped at ep {ep}, best val={best:.4f} (ep {int(np.argmin(va_hist))+1})')
    del tl, vl
    return model, tr_hist, va_hist


@torch.no_grad()
def predict_scaled(ds, model):
    '''Return (N, DECODER_LEN, 3) quantile predictions in SCALED delta units.
    Index comes from the Dataset, not from batch arithmetic — v2 reconstructed it
    as bi*BATCH_SIZE+i, which silently corrupts if the loader batch size differs.'''
    model.eval()
    out = np.zeros((len(ds), DECODER_LEN, len(QUANTILES)), np.float32)
    for sc, sx, px, fx, _, _, idx in DataLoader(ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=0):
        p, _, _ = model(sc.to(DEVICE), sx.to(DEVICE), px.to(DEVICE), fx.to(DEVICE))
        out[idx.numpy()] = p.cpu().numpy()
    return out


_m = build_model(0)
NPARAM = sum(p.numel() for p in _m.parameters())
del _m; gc.collect()
print(f'Model parameters: {NPARAM:,}')
print(f'  v2 REPORTED 205,173 but that config actually builds 1,231,159 — the figure in')
print(f'  PROD_V1_vs_V2.md is wrong by ~6x. This is a real reduction, not a re-labelling.')

In [ ]:
# ── Reconstruction, shrinkage, conformal, metrics ────────────────────────────
def to_metres(ds, qs):
    '''Scaled quantile output -> quantile DELTAS in metres. (N, T, 3)'''
    return qs * ds.sd_mat[:, :, None]


def band_of(step, size=BAND_SIZE):
    return min(step // size, (DECODER_LEN - 1) // size)


def fit_lambda(ds, qd_m):
    '''M1 — shrinkage toward persistence, fitted on the CALIBRATION block.

        level_hat = anchor + lambda * delta_hat          lambda in [0, 1]

    lambda = 0 reproduces persistence exactly, lambda = 1 is the raw model. The
    unshrunk model is inside the search space, so on the calibration data this is
    a one-parameter convex hull that cannot be worse than either endpoint — and
    it generalises because it is a single number per (district, horizon band).

    Per-district lambdas are pulled toward the global one in proportion to how
    little data supports them (hierarchical shrinkage, strength LAMBDA_PRIOR_N),
    so a district with 40 calibration points cannot pick an extreme value.

    This is what stops any district being catastrophically worse than doing
    nothing — v2's Aurangabad sat at MASE 2.84 with no floor.
    '''
    if len(ds) == 0:
        return {}
    mid = qd_m[:, :, 1]
    err = ds.actual - ds.anchor[:, None]          # true delta, metres
    ok = (ds.M > 0) & np.isfinite(err)
    bands = np.array([band_of(s) for s in range(DECODER_LEN)])

    def best_lambda(mask):
        if mask.sum() < 10: return None, 0
        e, p = err[mask], mid[mask]
        losses = [np.mean(np.abs(e - lam*p)) for lam in LAMBDA_GRID]
        return float(LAMBDA_GRID[int(np.argmin(losses))]), int(mask.sum())

    lam_global = {}
    for b in np.unique(bands):
        m = ok & (bands[None, :] == b)
        lam_global[b] = best_lambda(m)[0] or 1.0

    out = {}
    for dist in np.unique(ds.district):
        dm = (ds.district == dist)[:, None]
        for b in np.unique(bands):
            m = ok & dm & (bands[None, :] == b)
            lam_d, n = best_lambda(m)
            g = lam_global[b]
            lam = g if lam_d is None else (n*lam_d + LAMBDA_PRIOR_N*g) / (n + LAMBDA_PRIOR_N)
            out[(dist, int(b))] = float(np.clip(lam, 0.0, 1.0))
    out['_global'] = {int(b): float(v) for b, v in lam_global.items()}
    return out


def apply_lambda(ds, qd_m, lam):
    '''Recentre the median toward persistence, PRESERVING interval width.
    Shrinking the spread too would destroy coverage — the uncertainty is real
    even when the point forecast should be pulled back to 'no change'.'''
    if not lam: return qd_m
    out = qd_m.copy()
    mid = qd_m[:, :, 1]
    lo_half, hi_half = mid - qd_m[:, :, 0], qd_m[:, :, 2] - mid
    L = np.ones_like(mid)
    for j, dist in enumerate(ds.district):
        for s in range(DECODER_LEN):
            L[j, s] = lam.get((dist, band_of(s)), lam.get('_global', {}).get(band_of(s), 1.0))
    new_mid = L * mid
    out[:, :, 0] = new_mid - lo_half
    out[:, :, 1] = new_mid
    out[:, :, 2] = new_mid + hi_half
    return out


def fit_conformal(ds, qd_m, target=TARGET_COVERAGE):
    '''M4 — conformal margins from the CALIBRATION block, never from test.

    v2 computed the nonconformity quantile from the very test rows it then
    widened, so "80% after calibration" was a mathematical identity that any
    model would achieve. Here the scores come from data the model never trained
    on and never early-stopped on, so the calibrated coverage reported on test is
    a genuine out-of-sample number — and it will not land exactly on 80%.

    Split conformal assumes exchangeability, which temporal dependence violates,
    so treat the guarantee as approximate rather than exact.
    '''
    if len(ds) == 0: return {}
    lo = qd_m[:, :, 0] + ds.anchor[:, None]
    hi = qd_m[:, :, 2] + ds.anchor[:, None]
    score = np.maximum(lo - ds.actual, ds.actual - hi)
    ok = (ds.M > 0) & np.isfinite(score)
    bands = np.array([band_of(s) for s in range(DECODER_LEN)])

    def marg(mask):
        v = score[mask]
        if len(v) < 5: return None
        lvl = min(np.ceil((len(v)+1)*target)/len(v), 1.0)
        return float(np.quantile(v, lvl))

    out = {'_global': {}}
    for b in np.unique(bands):
        out['_global'][int(b)] = marg(ok & (bands[None, :] == b)) or 0.0
    for dist in np.unique(ds.district):
        dm = (ds.district == dist)[:, None]
        for b in np.unique(bands):
            m = ok & dm & (bands[None, :] == b)
            out[(dist, int(b))] = marg(m) if m.sum() >= CONF_MIN_N else out['_global'][int(b)]
    return out


def apply_conformal(ds, qd_m, margins):
    if not margins: return qd_m
    out = qd_m.copy()
    for j, dist in enumerate(ds.district):
        for s in range(DECODER_LEN):
            b = band_of(s)
            m = margins.get((dist, b))
            if m is None: m = margins.get('_global', {}).get(b, 0.0)
            out[j, s, 0] -= m
            out[j, s, 2] += m
    return out


print('to_metres / fit_lambda / apply_lambda / fit_conformal / apply_conformal defined.')

In [ ]:
# ── Turning window arrays into scored rows (daily + weekly) ──────────────────
def rows_daily(ds, qd_m, model_name, fold):
    '''One row per (window, evaluated horizon) where the target day was observed.'''
    recs = []
    for s in EVAL_STEPS:
        m = ds.M[:, s] > 0
        if not m.any(): continue
        a = ds.anchor[m]
        recs.append(pd.DataFrame({
            'model': model_name, 'fold': fold, 'district': ds.district[m],
            'date': ds.date.values[m], 'horizon': HNAME[s], 'step': s,
            'actual': ds.actual[m, s],
            'q10': a + qd_m[m, s, 0], 'q50': a + qd_m[m, s, 1], 'q90': a + qd_m[m, s, 2],
            'persistence': a,
        }))
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()


def rows_weekly(ds, qd_m, model_name, fold, win=7, min_obs=4):
    '''M11 — the 7-day MEAN level over the week ENDING at the horizon.

    t+7 -> mean over days t+1..t+7, t+14 -> t+8..t+14, t+30 -> t+24..t+30.

    Why this exists: the daily district median is, for several districts, two
    wells averaged on one day. The median real 7-day movement is 0.06-0.14 m in
    Raigad / Bhandara / Beed / Pune — smaller than the measurement noise, so no
    model can beat persistence there on a single day. In the ridge probe, moving
    to a weekly mean took districts-with-skill at t+7 from 3 to 9. It is also the
    quantity a farmer or a district officer actually acts on.

    Reported ALONGSIDE the daily target, never instead of it.
    '''
    recs = []
    for s in EVAL_STEPS:
        lo = max(0, s - win + 1)
        sl = slice(lo, s + 1)
        mk = ds.M[:, sl]
        nobs = mk.sum(1)
        keep = nobs >= min_obs
        if not keep.any(): continue
        act = np.where(mk > 0, ds.actual[:, sl], np.nan)
        act_mean = np.nanmean(act[keep], axis=1)
        a = ds.anchor[keep]
        q = qd_m[keep][:, sl, :].mean(axis=1)
        recs.append(pd.DataFrame({
            'model': model_name, 'fold': fold, 'district': ds.district[keep],
            'date': ds.date.values[keep], 'horizon': HNAME[s], 'step': s,
            'actual': act_mean,
            'q10': a + q[:, 0], 'q50': a + q[:, 1], 'q90': a + q[:, 2],
            'persistence': a,
        }))
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()


def pinball(a, q10, q50, q90):
    tot = 0.0
    for q, p in [(0.1, q10), (0.5, q50), (0.9, q90)]:
        d = a - p
        tot += np.mean(np.maximum(q*d, (q-1)*d))
    return tot / 3.0


def _clim_mse(g, a):
    '''Variance reference for NSE.

    Pooling districts and using np.var(actual) makes NSE meaningless here: levels
    run from -3 m (Raigad) to -45 m (Dhule), so the between-district spread
    dominates and every model scores ~0.99. The correct reference is each
    district's own mean level - the climatological forecast within that district.
    '''
    if 'district' in g.columns and g['district'].nunique() > 1:
        clim = g.groupby('district')['actual'].transform('mean').values
    else:
        clim = a.mean()
    return float(np.mean((a - clim) ** 2))


def metrics(pred, by=None):
    '''M10 — MASE is the decision metric, skill score is the explainable one,
    MAPE is included because it is easy to explain, with the datum caveat.'''
    keys = (['horizon'] if by is None else [by, 'horizon'])
    rows = []
    for k, g in pred.groupby(keys, sort=False):
        k = k if isinstance(k, tuple) else (k,)
        a, p, nv = g['actual'].values, g['q50'].values, g['persistence'].values
        mae, mae_n = np.mean(np.abs(a-p)), np.mean(np.abs(a-nv))
        mse, mse_n = np.mean((a-p)**2), np.mean((a-nv)**2)
        r = {
            'MAE (m)':      round(float(mae), 4),
            'RMSE (m)':     round(float(np.sqrt(mse)), 4),
            'Persist MAE':  round(float(mae_n), 4),
            'MASE':         round(float(mae/mae_n), 3) if mae_n > 1e-9 else np.nan,
            'Skill %':      round(float((1 - mse/mse_n)*100), 1) if mse_n > 1e-12 else np.nan,
            'MAPE (%)':     round(float(np.mean(np.abs((a-p)/np.maximum(np.abs(a), 0.5)))*100), 2),
            'NSE':          round(float(1 - mse/max(_clim_mse(g, a), 1e-12)), 3),
            'Pinball':      round(float(pinball(a, g['q10'].values, g['q50'].values, g['q90'].values)), 4),
            'Cov (%)':      round(float(((a >= g['q10'].values) & (a <= g['q90'].values)).mean()*100), 1),
            'Width (m)':    round(float((g['q90'].values - g['q10'].values).mean()), 3),
            'n':            len(g),
        }
        if by is None: r['Horizon'] = k[0]
        else:          r[by], r['Horizon'] = k[0], k[1]
        rows.append(r)
    out = pd.DataFrame(rows)
    order = ['t+7','t+14','t+30']
    out['_o'] = out['Horizon'].map({h:i for i,h in enumerate(order)})
    return out.sort_values(([by] if by else []) + ['_o']).drop(columns='_o').reset_index(drop=True)


print('rows_daily / rows_weekly / metrics defined.')

In [ ]:
# ── Baselines (M7) — same windows, same targets, so MASE is comparable ───────
def baseline_climatology(train_ds, ds):
    '''Anchor + the average change seen at this (district, calendar month, step)
    during training. A seasonal naive that knows the monsoon exists.'''
    qd = np.zeros((len(ds), DECODER_LEN, 3), np.float32)
    if len(train_ds) == 0: return qd
    tr_delta = train_ds.actual - train_ds.anchor[:, None]
    tr_month = pd.Series(train_ds.date).dt.month.values
    tbl = {}
    for dist in np.unique(train_ds.district):
        for mo in range(1, 13):
            m = (train_ds.district == dist) & (tr_month == mo)
            if m.sum() >= 3:
                d = np.where(train_ds.M[m] > 0, tr_delta[m], np.nan)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    tbl[(dist, mo)] = np.nan_to_num(np.nanmean(d, axis=0))
    months = pd.Series(ds.date).dt.month.values
    for j in range(len(ds)):
        v = tbl.get((ds.district[j], months[j]))
        if v is not None:
            qd[j, :, 1] = v
    qd[:, :, 0] = qd[:, :, 1] - 1.0
    qd[:, :, 2] = qd[:, :, 1] + 1.0
    return qd


def _fit_predict_tabular(train_ds, ds, kind='ridge'):
    '''Ridge / LightGBM on the anchor-day feature snapshot, one model per
    evaluated horizon, predicting the same Δ the TFT predicts.

    This is the comparison a reviewer will ask for: is the transformer earning
    its keep over a well-specified regression and a gradient-boosted tree on
    ~2,900 training windows? Cheap to run and you want the answer before the viva.
    '''
    qd = np.zeros((len(ds), DECODER_LEN, 3), np.float32)
    if len(train_ds) < 50 or len(ds) == 0: return qd
    Xtr_all = np.hstack([train_ds.tab, train_ds.did[:, None].astype(np.float32)])
    Xte_all = np.hstack([ds.tab,       ds.did[:, None].astype(np.float32)])

    for s in EVAL_STEPS:
        mtr = train_ds.M[:, s] > 0
        if mtr.sum() < 50: continue
        ytr = (train_ds.actual[mtr, s] - train_ds.anchor[mtr])
        Xtr, Xte = Xtr_all[mtr], Xte_all
        if kind == 'ridge':
            mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
            pred = {q: None for q in QUANTILES}
            m = Ridge(alpha=10.0).fit((Xtr-mu)/sd, ytr)
            mid = m.predict((Xte-mu)/sd)
            res = np.quantile(np.abs(ytr - m.predict((Xtr-mu)/sd)), 0.8)
            qd[:, s, 1] = mid; qd[:, s, 0] = mid - res; qd[:, s, 2] = mid + res
        else:
            for qi, q in enumerate(QUANTILES):
                if HAS_LGB:
                    mdl = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=300,
                                            learning_rate=0.05, num_leaves=15, min_child_samples=30,
                                            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                                            verbose=-1)
                else:
                    mdl = HistGradientBoostingRegressor(loss='quantile', quantile=q,
                                                        max_iter=300, learning_rate=0.05,
                                                        max_leaf_nodes=15, min_samples_leaf=30)
                mdl.fit(Xtr, ytr)
                qd[:, s, qi] = mdl.predict(Xte)
        # enforce monotonicity for the tabular models too
        qd[:, s, 0] = np.minimum(qd[:, s, 0], qd[:, s, 1])
        qd[:, s, 2] = np.maximum(qd[:, s, 2], qd[:, s, 1])
    return qd


print(f'Baselines ready: persistence, climatology-of-change, ridge, '
      f'{"LightGBM" if HAS_LGB else "HistGradientBoosting"}.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# WALK-FORWARD DRIVER
#   train  : ... -> fit_end                     stride 7
#   es     : early stopping only                stride 2   (first VAL_ES_FRAC of the val span)
#   cal    : lambda + conformal only            stride 2   (rest of the val span; training never sees it)
#   test   : never seen by anything             stride 1
# Per-fold results are written to disk as they complete, so a Colab disconnect
# costs one fold rather than the whole run.
# ══════════════════════════════════════════════════════════════════════════
from dateutil.relativedelta import relativedelta

daily_parts, weekly_parts, base_parts = [], [], []
fold_curves, fold_lambdas, fold_models, window_log = {}, {}, {}, []
t_start = time.time()

for fold in FOLDS:
    name = fold['name']
    cache = OUT_DIR / f'preds_{name}.pkl'
    if cache.exists():
        blob = pd.read_pickle(cache)
        daily_parts.append(blob['daily']); weekly_parts.append(blob['weekly'])
        base_parts.append(blob['base']);   fold_curves[name] = blob['curves']
        fold_lambdas[name] = blob['lam'];  window_log.append(blob['win'])
        print(f'[{name}] loaded from cache — delete {cache} to recompute')
        continue

    test_start, test_end = fold['test_start'], fold['test_end']
    train_end = pd.Timestamp(test_start) - pd.Timedelta(days=1)
    val_start = train_end - relativedelta(months=VAL_MONTHS) + pd.Timedelta(days=1)
    fit_end   = val_start - pd.Timedelta(days=1)
    span      = (train_end - val_start).days
    es_end    = val_start + pd.Timedelta(days=int(span * VAL_ES_FRAC))
    cal_start = es_end + pd.Timedelta(days=1)

    print(f'\n{"="*80}\nFOLD: {name}   [{time.time()-t_start:.0f}s elapsed]')
    print(f'  train : ...        -> {fit_end.date()}    stride {TRAIN_STRIDE}')
    print(f'  es    : {val_start.date()} -> {es_end.date()}    stride {VAL_STRIDE}  (early stopping only)')
    print(f'  cal   : {cal_start.date()} -> {train_end.date()}    stride {VAL_STRIDE}  (lambda + conformal only)')
    print(f'  test  : {test_start} -> {test_end}    stride {TEST_STRIDE}  (never seen)')
    print('='*80)

    dfn, sd_enc, sd_h = normalize_for_fold(df, fit_end)
    mk = lambda ss, se, st: GWLDataset(dfn, PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT,
                                       sd_enc, sd_h, split_start=ss, split_end=se, stride=st)
    train_ds = mk(None,                  str(fit_end.date()),   TRAIN_STRIDE)
    es_ds    = mk(str(val_start.date()), str(es_end.date()),    VAL_STRIDE)
    cal_ds   = mk(str(cal_start.date()), str(train_end.date()), VAL_STRIDE)
    test_ds  = mk(test_start,            test_end,              TEST_STRIDE)

    wl = {'fold': name, 'train': len(train_ds), 'es': len(es_ds),
          'cal': len(cal_ds), 'test': len(test_ds),
          'drop_enc': train_ds.n_drop_enc, 'drop_dec': train_ds.n_drop_dec,
          'drop_anchor': train_ds.n_drop_anchor}
    window_log.append(wl)
    print(f'  windows: train={wl["train"]:,}  es={wl["es"]:,}  cal={wl["cal"]:,}  test={wl["test"]:,}')
    print(f'  params/independent-window ~= {NPARAM/max(wl["train"]/7,1):,.0f}   '
          f'(stride-7 windows still share {ENCODER_LEN+DECODER_LEN-7}/{ENCODER_LEN+DECODER_LEN} days)')
    per_d = pd.Series(train_ds.counts()).sort_values()
    print(f'  train windows per district: min={per_d.iloc[0]} ({per_d.index[0]}), '
          f'max={per_d.iloc[-1]} ({per_d.index[-1]}), ratio={per_d.iloc[-1]/max(per_d.iloc[0],1):.1f}x')

    # ── seed ensemble ────────────────────────────────────────────────────────
    acc_cal = np.zeros((len(cal_ds), DECODER_LEN, 3), np.float64)
    acc_test = np.zeros((len(test_ds), DECODER_LEN, 3), np.float64)
    curves = []
    for si, seed in enumerate(SEEDS):
        print(f'  -- seed {seed} ({si+1}/{len(SEEDS)}) --')
        model, tr_h, va_h = train_one(f'{name}|s{seed}', train_ds, es_ds, seed)
        curves.append((tr_h, va_h))
        acc_cal  += predict_scaled(cal_ds,  model)
        acc_test += predict_scaled(test_ds, model)
        if si == len(SEEDS) - 1:
            fold_models[name] = model.cpu()
        else:
            del model
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    fold_curves[name] = curves

    cal_qd  = to_metres(cal_ds,  (acc_cal  / len(SEEDS)).astype(np.float32))
    test_qd = to_metres(test_ds, (acc_test / len(SEEDS)).astype(np.float32))

    # ── M1 shrinkage, then M4 conformal — both fitted on `cal`, applied to test ─
    lam = fit_lambda(cal_ds, cal_qd)
    fold_lambdas[name] = lam
    lam_vals = [v for k, v in lam.items() if k != '_global']
    print(f'  lambda: global={lam.get("_global")}, '
          f'district range [{min(lam_vals):.2f}, {max(lam_vals):.2f}], mean {np.mean(lam_vals):.2f}')

    cal_qd_s  = apply_lambda(cal_ds,  cal_qd,  lam)
    test_qd_s = apply_lambda(test_ds, test_qd, lam)
    margins   = fit_conformal(cal_ds, cal_qd_s)
    test_qd_c = apply_conformal(test_ds, test_qd_s, margins)

    # ── collect rows ─────────────────────────────────────────────────────────
    d_raw  = rows_daily(test_ds, test_qd,   'TFT (raw)',        name)
    d_shr  = rows_daily(test_ds, test_qd_s, 'TFT',              name)
    d_cal  = rows_daily(test_ds, test_qd_c, 'TFT (calibrated)', name)
    daily = pd.concat([d_raw, d_shr, d_cal], ignore_index=True)
    weekly = rows_weekly(test_ds, test_qd_s, 'TFT', name)

    bl = []
    for bname, qd in [
        ('Climatology', baseline_climatology(train_ds, test_ds)),
        ('Ridge',       _fit_predict_tabular(train_ds, test_ds, 'ridge')),
        ('LightGBM',    _fit_predict_tabular(train_ds, test_ds, 'gbm')),
    ]:
        bl.append(rows_daily(test_ds, qd, bname, name))
    base = pd.concat([b for b in bl if len(b)], ignore_index=True)

    daily_parts.append(daily); weekly_parts.append(weekly); base_parts.append(base)
    pd.to_pickle({'daily': daily, 'weekly': weekly, 'base': base,
                  'curves': curves, 'lam': lam, 'win': wl}, cache)

    print(f'\n[{name}] TFT (shrunk) test metrics:')
    print(metrics(d_shr).set_index('Horizon').to_string())

    del train_ds, es_ds, cal_ds, test_ds, dfn
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

DAILY  = pd.concat(daily_parts,  ignore_index=True)
WEEKLY = pd.concat(weekly_parts, ignore_index=True)
BASE   = pd.concat(base_parts,   ignore_index=True)
ALL    = pd.concat([DAILY, BASE], ignore_index=True)
ALL.to_csv(OUT_DIR/'predictions_all.csv', index=False)
WEEKLY.to_csv(OUT_DIR/'predictions_weekly.csv', index=False)
print(f'\nTotal wall time: {(time.time()-t_start)/60:.1f} min')
print(f'Saved -> outputs/predictions_all.csv ({len(ALL):,} rows)')
print(pd.DataFrame(window_log).set_index('fold').to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# HEADLINE
# ══════════════════════════════════════════════════════════════════════════
def headline_subset(d):
    x = d.copy()
    x['tier'] = x['district'].map(TIER)
    x['low']  = x['district'].map(LOW_STATION)
    if HEADLINE_EXCLUDE_LOW_STATION:
        x = x[x['low'] == 0]
    return x

tft = headline_subset(DAILY[DAILY.model == 'TFT'])
print('='*90)
print('HEADLINE — TFT, all folds pooled, single-well districts excluded, DAILY target')
print('='*90)
head = metrics(tft).set_index('Horizon')
print(head.to_string())

nwin = int((head['MASE'] < 1).sum())
print(f'\n>>> Beats persistence (MASE < 1) at {nwin}/{len(head)} horizons')
for h in head.index:
    r = head.loc[h]
    print(f'    {h:<5}  MASE {r["MASE"]:.3f}   skill {r["Skill %"]:+.1f}%   '
          f'MAE {r["MAE (m)"]:.3f} m vs persistence {r["Persist MAE"]:.3f} m')
if nwin == len(head):
    print('>>> Genuine skill over the naive forecast at every horizon.')
elif nwin == 0:
    print('>>> Still does NOT beat persistence. Do not tune further — revisit the framing.')
else:
    print('>>> Skill at some horizons only. Report those; be explicit about the rest.')

print('\n' + '='*90)
print('PER-FOLD / PER-SEASON')
print('='*90)
per_fold = metrics(tft, by='fold')
print(per_fold.to_string(index=False))
print('''
  fold1 is the hardest by construction: it trains on ~18 months, of which 2023 is
  only 57% observed state-wide. Accuracy improving across folds is the training-record
  effect, not seasonal luck — see the record-length plot below.''')

head.to_csv(OUT_DIR/'metrics_headline.csv')
per_fold.to_csv(OUT_DIR/'metrics_per_fold.csv', index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BASELINE LADDER (M7) — is the transformer earning its keep?
# ══════════════════════════════════════════════════════════════════════════
ladder = []
for mname in ['Climatology', 'Ridge', 'LightGBM', 'TFT (raw)', 'TFT']:
    sub = headline_subset(ALL[ALL.model == mname])
    if not len(sub): continue
    m = metrics(sub).set_index('Horizon')
    for h in m.index:
        ladder.append({'model': mname, 'horizon': h, 'MASE': m.loc[h,'MASE'],
                       'MAE (m)': m.loc[h,'MAE (m)'], 'Skill %': m.loc[h,'Skill %'],
                       'MAPE (%)': m.loc[h,'MAPE (%)'], 'Pinball': m.loc[h,'Pinball']})
L = pd.DataFrame(ladder)
print('='*90)
print('BASELINE LADDER — identical windows, identical targets, so MASE is directly comparable')
print('='*90)
piv = L.pivot(index='model', columns='horizon', values='MASE')
piv = piv[[c for c in ['t+7','t+14','t+30'] if c in piv.columns]]
piv.loc['Persistence'] = 1.0
order = ['Persistence','Climatology','Ridge','LightGBM','TFT (raw)','TFT']
print(piv.reindex([o for o in order if o in piv.index]).round(3).to_string())
print('\n(MASE — lower is better; 1.000 is the persistence baseline by definition)')
sk = L.pivot(index='model', columns='horizon', values='Skill %')
sk = sk[[c for c in ['t+7','t+14','t+30'] if c in sk.columns]]
sk.loc['Persistence'] = 0.0
print(sk.reindex([o for o in order if o in sk.index]).round(1).to_string())
print('\n(Skill % — "reduces squared error by X% versus assuming no change")')

mp = L.pivot(index='model', columns='horizon', values='MAPE (%)')
mp = mp[[c for c in ['t+7','t+14','t+30'] if c in mp.columns]]
print(mp.reindex([o for o in order if o in mp.index and o != 'Persistence']).round(2).to_string())
print('\n(MAPE % — no Persistence row: MAPE has no trivial self-referential value the way MASE=1')
print('does. Datum-sensitive — a district near 0m GWL inflates MAPE regardless of model quality.')
print('Report next to MASE, never alone; see the caveat in V3_PLAN.md / PROD_V1_vs_V2.md.)')

best = piv.drop(index='Persistence').idxmin()
print('\nBest model per horizon:', dict(best))
if all(v.startswith('TFT') for v in best.values):
    print('>>> The TFT wins at every horizon against ridge, GBM and climatology.')
else:
    print('>>> The TFT does NOT win everywhere. Report this honestly — a simpler model')
    print('    winning at some horizon is a finding, not a failure of the project.')
L.to_csv(OUT_DIR/'baseline_ladder.csv', index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PER-DISTRICT, WITH DATA TIER ALONGSIDE
# ══════════════════════════════════════════════════════════════════════════
pd_tft = metrics(headline_subset(DAILY[DAILY.model=='TFT']), by='district')
pd_rdg = metrics(headline_subset(BASE[BASE.model=='Ridge']), by='district')[['district','Horizon','MASE']]
pd_rdg = pd_rdg.rename(columns={'MASE':'ridge MASE'})
per_dist = pd_tft.merge(pd_rdg, on=['district','Horizon'], how='left')
per_dist['tier'] = per_dist['district'].map(TIER)

cols = ['district','tier','MAE (m)','Persist MAE','MASE','ridge MASE','Skill %',
        'MAPE (%)','Cov (%)','Width (m)','n']
for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist.Horizon==h][cols].sort_values('MASE').reset_index(drop=True)
    print(f'--- {h}  (MASE < 1 beats persistence) ---')
    print(sub.to_string(index=False))
    print(f'   beating persistence : {(sub["MASE"]<1).sum()}/{len(sub)}')
    print(f'   MEDIAN  MASE={sub["MASE"].median():.3f}  MAE={sub["MAE (m)"].median():.3f} m  '
          f'skill={sub["Skill %"].median():+.1f}%  coverage={sub["Cov (%)"].median():.1f}%')
    for t in ['A','B','C']:
        s = sub[sub.tier==t]
        if len(s):
            print(f'     tier {t}: median MASE {s["MASE"].median():.3f}  '
                  f'({(s["MASE"]<1).sum()}/{len(s)} beat persistence)')
    print()

print('''INTERPRETATION
  A district where BOTH the TFT and the ridge lose to persistence is a data
  limitation, not a model bug — that is why the ridge column is here. In the v2
  audit, 6 of the 7 TFT failures also failed for ridge (Aurangabad, Bhandara,
  Nashik, Jalgaon, Dhule, Kolhapur), and all are thin-panel / gappy-2023 tier C.
  A district where ridge succeeds but the TFT fails is a genuine modelling
  target — Ratnagiri was the only such case in v2.''')

per_dist.to_csv(OUT_DIR/'metrics_per_district.csv', index=False)

low = DAILY[(DAILY.model=='TFT') & (DAILY.district.map(LOW_STATION)==1)]
if len(low):
    print('\n' + '='*90)
    print('SINGLE-WELL DISTRICTS — reported separately, NOT in the headline')
    print('='*90)
    print(metrics(low, by='district').to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# IS THE IMPROVEMENT STATISTICALLY REAL? (M10)
#   Diebold-Mariano with Newey-West HAC variance, vs persistence, per horizon.
#   Plus a district-level bootstrap CI on the median MASE.
# ══════════════════════════════════════════════════════════════════════════
def dm_test(a, p, nv, lag):
    '''H0: model and persistence have equal expected absolute error.
    d_t = |e_model| - |e_naive|; negative mean => model is better.
    HAC-corrected because overlapping forecast windows are autocorrelated.'''
    d = np.abs(a-p) - np.abs(a-nv)
    n = len(d)
    if n < 30: return np.nan, np.nan
    dbar = d.mean(); dc = d - dbar
    gamma0 = np.mean(dc*dc)
    var = gamma0
    for l in range(1, min(lag, n-1)+1):
        g = np.mean(dc[l:]*dc[:-l])
        var += 2*(1 - l/(lag+1))*g
    var = max(var, 1e-12)
    stat = dbar / np.sqrt(var/n)
    from math import erf, sqrt
    pval = 2*(1 - 0.5*(1+erf(abs(stat)/sqrt(2))))
    return float(stat), float(pval)

print('='*90)
print('DIEBOLD-MARIANO vs PERSISTENCE  (negative stat = TFT better)')
print('='*90)
rows = []
for h, lag in [('t+7',7), ('t+14',14), ('t+30',30)]:
    g = headline_subset(DAILY[(DAILY.model=='TFT') & (DAILY.horizon==h)]).sort_values(['district','date'])
    s, p = dm_test(g['actual'].values, g['q50'].values, g['persistence'].values, lag)
    rows.append({'horizon': h, 'n': len(g), 'DM stat': round(s,3), 'p-value': f'{p:.2e}',
                 'verdict': ('TFT better' if s < 0 else 'persistence better') +
                            (' (p<0.05)' if p < 0.05 else ' (not significant)')})
dm = pd.DataFrame(rows)
print(dm.to_string(index=False))

print('\n' + '='*90)
print('BOOTSTRAP 95% CI on median per-district MASE  (resampling DISTRICTS, 2000 draws)')
print('='*90)
rng = np.random.default_rng(0)
rows = []
for h in ['t+7','t+14','t+30']:
    v = per_dist[per_dist.Horizon==h].set_index('district')['MASE'].dropna()
    if len(v) < 3: continue
    draws = [np.median(rng.choice(v.values, size=len(v), replace=True)) for _ in range(2000)]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    rows.append({'horizon': h, 'median MASE': round(float(v.median()),3),
                 '95% CI': f'[{lo:.3f}, {hi:.3f}]',
                 'CI excludes 1.0': 'YES' if hi < 1.0 else 'no'})
bs = pd.DataFrame(rows)
print(bs.to_string(index=False))
dm.to_csv(OUT_DIR/'significance_dm.csv', index=False)
bs.to_csv(OUT_DIR/'significance_bootstrap.csv', index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# WEEKLY-MEAN TARGET (M11) — the operational quantity, reported alongside daily
# ══════════════════════════════════════════════════════════════════════════
wk = headline_subset(WEEKLY)
print('='*90)
print('7-DAY-MEAN water table (mean level over the week ENDING at the horizon)')
print('='*90)
wm = metrics(wk).set_index('Horizon')
print(wm.to_string())

print('\nDaily vs weekly-mean, side by side (MASE):')
cmp = pd.DataFrame({'daily': head['MASE'], 'weekly-mean': wm['MASE']})
cmp['change'] = (cmp['weekly-mean'] - cmp['daily']).round(3)
print(cmp.to_string())

wpd = metrics(wk, by='district')
for h in ['t+7','t+14','t+30']:
    s = wpd[wpd.Horizon==h]
    d = per_dist[per_dist.Horizon==h]
    print(f'  {h}: districts beating persistence — daily {(d["MASE"]<1).sum()}/{len(d)}, '
          f'weekly-mean {(s["MASE"]<1).sum()}/{len(s)}')

print('''
  Why both are reported: for several districts the daily district median is two
  wells averaged on one day, and the median real 7-day movement is 0.06-0.14 m —
  below the measurement noise, so t+7 is unbeatable on a daily basis by anyone.
  The weekly mean is the quantity the dashboards should show. The daily numbers
  stay in the report so the v2 -> v3 comparison remains like-for-like.''')
wm.to_csv(OUT_DIR/'metrics_weekly.csv')
wpd.to_csv(OUT_DIR/'metrics_weekly_per_district.csv', index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# UNCERTAINTY — honest calibration (M4)
# ══════════════════════════════════════════════════════════════════════════
print('='*90)
print('COVERAGE — raw model / after shrinkage / after conformal')
print('  Conformal margins come from the CALIBRATION block, NOT from these test rows.')
print('  v2 fitted them on the test set itself, which forces 80% by construction.')
print('  Landing NEAR but not exactly on 80% here is the correct, honest outcome.')
print('='*90)
rows = []
for mname in ['TFT (raw)', 'TFT', 'TFT (calibrated)']:
    s = headline_subset(DAILY[DAILY.model==mname])
    for h in ['t+7','t+14','t+30']:
        g = s[s.horizon==h]
        if not len(g): continue
        cov = ((g.actual>=g.q10)&(g.actual<=g.q90)).mean()*100
        rows.append({'model': mname, 'horizon': h, 'coverage %': round(cov,1),
                     'below q10 %': round((g.actual<g.q10).mean()*100,1),
                     'above q90 %': round((g.actual>g.q90).mean()*100,1),
                     'width (m)': round((g.q90-g.q10).mean(),3)})
cov_tbl = pd.DataFrame(rows)
print(cov_tbl.to_string(index=False))
print('\n  (target: 80% coverage, 10% below q10, 10% above q90 — asymmetry means bias)')

cal_sub = headline_subset(DAILY[DAILY.model=='TFT (calibrated)'])
cov_d = (cal_sub.assign(c=(cal_sub.actual>=cal_sub.q10)&(cal_sub.actual<=cal_sub.q90))
         .groupby(['district','horizon'])['c'].mean().mul(100).round(1).unstack('horizon'))
cov_d.columns.name = None
cov_d['tier'] = cov_d.index.map(TIER)
print('\nPer-district coverage after calibration (%):')
print(cov_d.to_string())
cov_tbl.to_csv(OUT_DIR/'coverage.csv', index=False)
cov_d.to_csv(OUT_DIR/'coverage_by_district.csv')

print('\n' + '='*90)
print('SHRINKAGE WEIGHTS lambda (M1)   0 = pure persistence, 1 = raw model')
print('='*90)
lam_rows = []
for fname, lam in fold_lambdas.items():
    for k, v in lam.items():
        if k == '_global': continue
        lam_rows.append({'fold': fname, 'district': k[0], 'band': f'steps {k[1]*BAND_SIZE}-{k[1]*BAND_SIZE+BAND_SIZE-1}', 'lambda': v})
LAM = pd.DataFrame(lam_rows)
if len(LAM):
    pv = LAM.pivot_table(index='district', columns='band', values='lambda', aggfunc='mean').round(2)
    pv['tier'] = pv.index.map(TIER)
    print(pv.sort_values(pv.columns[0]).to_string())
    print('''
  A low lambda means the calibration block said 'this district's model signal is
  not worth trusting — fall back toward no-change'. Those are exactly the tier C
  districts. This is the mechanism that stops any district blowing up the way
  Aurangabad did in v2 (MASE 2.84 at t+7 with no floor).''')
    LAM.to_csv(OUT_DIR/'lambda_weights.csv', index=False)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# REGRESSION GATE (M14) — nothing may silently get worse
# ══════════════════════════════════════════════════════════════════════════
if V2_METRICS.exists():
    v2 = pd.read_csv(V2_METRICS)
    v2 = v2.rename(columns={c: c.strip() for c in v2.columns})
    hcol = 'Horizon' if 'Horizon' in v2.columns else 'horizon'
    has_mape_v2 = 'MAPE (%)' in v2.columns
    v2_cols = ['district', hcol, 'MASE'] + (['MAPE (%)'] if has_mape_v2 else [])
    v2 = v2[v2_cols].rename(columns={hcol: 'Horizon', 'MASE': 'MASE_v2', 'MAPE (%)': 'MAPE_v2'})
    keep_cols = ['district', 'Horizon', 'MASE', 'tier'] + (['MAPE (%)'] if has_mape_v2 else [])
    cmp = per_dist[keep_cols].merge(v2, on=['district', 'Horizon'], how='inner')
    cmp['delta']    = (cmp['MASE'] - cmp['MASE_v2']).round(3)
    cmp['change %'] = ((cmp['MASE']/cmp['MASE_v2'] - 1)*100).round(1)
    cmp['status']   = np.where(cmp['change %'] > 10, 'DEGRADED >10%',
                        np.where(cmp['change %'] < -10, 'improved >10%', 'unchanged'))
    if has_mape_v2:
        cmp['mape_delta'] = (cmp['MAPE (%)'] - cmp['MAPE_v2']).round(2)
    show_cols = ['district','tier','MASE_v2','MASE','change %','status']
    if has_mape_v2:
        show_cols = ['district','tier','MASE_v2','MASE','change %','MAPE_v2','MAPE (%)','mape_delta','status']
    for h in ['t+7','t+14','t+30']:
        s = cmp[cmp.Horizon==h].sort_values('change %')
        if not len(s): continue
        print(f'--- {h} ---')
        print(s[show_cols].to_string(index=False))
        print(f'   improved (MASE): {(s["change %"]<0).sum()}/{len(s)}   '
              f'median change: {s["change %"].median():+.1f}%')
        if has_mape_v2:
            print(f'   improved (MAPE): {(s["mape_delta"]<0).sum()}/{len(s)}   '
                  f'median MAPE change: {s["mape_delta"].median():+.2f} pp')
        bad = s[s.status=='DEGRADED >10%']
        if len(bad):
            print(f'   *** {len(bad)} DEGRADED >10% (by MASE): {list(bad.district)} — must be explained')
        print()
    cmp.to_csv(OUT_DIR/'regression_gate_v2_vs_v3.csv', index=False)
    print('NOTE: this comparison is only valid if v2 and v3 ran on the SAME master_dataset.csv.')
    print('      v1 -> v2 was not comparable because the data layer changed underneath it.')
else:
    print('No tft_metrics_per_district_v2.csv uploaded — regression gate skipped.')
    print("To enable: rename the v2 run's outputs/tft_metrics_per_district.csv")
    print("to tft_metrics_per_district_v2.csv and upload it in step 1.")

In [ ]:
# ── Training curves + overfitting check ──────────────────────────────────────
n = len(FOLDS); ncol = 3; nrow = int(np.ceil(n/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6*ncol, 4*nrow), sharey=True)
ax_flat = np.atleast_1d(axes).ravel()
for ax, fold in zip(ax_flat, FOLDS):
    for si, (tr, va) in enumerate(fold_curves[fold['name']]):
        ax.plot(tr, alpha=0.7, lw=1, color='C0', label='Train' if si==0 else None)
        ax.plot(va, alpha=0.7, lw=1, color='C1', label='Val'   if si==0 else None)
        ax.axvline(int(np.argmin(va)), color='red', ls='--', lw=0.8, alpha=0.5)
    ax.set_title(fold['name'], fontsize=10); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
ax_flat[0].set_ylabel('Masked pinball loss'); ax_flat[0].legend(fontsize=8)
for ax in ax_flat[n:]: ax.axis('off')
plt.tight_layout(); plt.savefig(OUT_DIR/'training_curves.png', dpi=150); plt.show()

print('Train/val gap at the best epoch — a large ratio means it is still memorising:')
for f in FOLDS:
    for si, (tr, va) in enumerate(fold_curves[f['name']]):
        b = int(np.argmin(va))
        print(f"  {f['name']:<24} seed{si}  best ep {b+1:>3} | val={va[b]:.4f} "
              f"train={tr[b]:.4f} | train/val={tr[b]/max(va[b],1e-9):.2f}")

# Accuracy vs how much training record was available — the argument for continuing
# to collect data, and the reason fold1 is hard.
days = {f['name']: (pd.Timestamp(f['test_start']) - df.date.min()).days for f in FOLDS}
pf = metrics(headline_subset(DAILY[DAILY.model=='TFT']), by='fold')
fig, ax = plt.subplots(figsize=(8,4.5))
for h, mk in [('t+7','o'), ('t+14','s'), ('t+30','^')]:
    s = pf[pf.Horizon==h]
    ax.plot([days[f] for f in s.fold], s['MASE'], marker=mk, label=h)
ax.axhline(1.0, color='k', ls='--', lw=1, label='persistence')
ax.set_xlabel('Days of training record available'); ax.set_ylabel('MASE')
ax.set_title('Accuracy vs length of training record'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR/'mase_vs_record_length.png', dpi=150); plt.show()

In [ ]:
# ── Per-district skill, coloured by data tier ────────────────────────────────
TIER_COLOR = {'A':'#1b7837', 'B':'#7fbf7b', 'C':'#d6604d', 'D':'#999999'}
for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist.Horizon==h].sort_values('MASE')
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(sub['district'], sub['MASE'], color=[TIER_COLOR.get(t,'#999') for t in sub['tier']])
    ax.axvline(1.0, color='black', ls='--', lw=1.5)
    ax.set_xlabel('MASE  (< 1 = better than assuming no change)')
    ax.set_title(f'Per-district skill @ {h} — bar colour = data tier')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    handles = [plt.Rectangle((0,0),1,1, color=c) for t,c in TIER_COLOR.items() if t in set(sub['tier'])]
    labels  = [f'tier {t}' for t in TIER_COLOR if t in set(sub['tier'])]
    ax.legend(handles, labels, loc='lower right')
    plt.tight_layout(); plt.savefig(OUT_DIR/f'mase_{h.replace("+","")}.png', dpi=150); plt.show()

In [ ]:
# ── Forecast vs actual, with persistence overlaid ────────────────────────────
tft_p = DAILY[DAILY.model=='TFT (calibrated)']
show = (per_dist[per_dist.Horizon=='t+30'].sort_values('MASE')['district'].head(2).tolist() +
        per_dist[per_dist.Horizon=='t+30'].sort_values('MASE')['district'].tail(2).tolist())
for dist in show:
    s = tft_p[(tft_p.district==dist) & (tft_p.horizon=='t+30')].sort_values('date')
    if not len(s): continue
    x = range(len(s))
    fig, ax = plt.subplots(figsize=(14,4))
    ax.fill_between(x, s['q10'], s['q90'], alpha=0.2, color='steelblue', label='80% interval')
    ax.plot(x, s['q50'],         color='steelblue', lw=1.5, label='TFT q50')
    ax.plot(x, s['persistence'], color='darkorange', lw=1.2, ls=':', label='persistence')
    ax.plot(x, s['actual'],      color='black',      lw=1.5, ls='--', label='actual')
    mase = per_dist[(per_dist.district==dist)&(per_dist.Horizon=='t+30')]['MASE'].iloc[0]
    ax.set_title(f'{dist} (tier {TIER.get(dist)}) — t+30, all test folds — MASE {mase:.3f}')
    ax.set_ylabel('GWL (m)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR/f'forecast_{dist.lower().replace(" ","_")}.png', dpi=150); plt.show()

In [ ]:
# ── Variable-selection importance (last fold, last seed) ─────────────────────
name = FOLDS[-1]['name']
model_final = fold_models[name].to(DEVICE).eval()
fit_end_f = (pd.Timestamp(FOLDS[-1]['test_start']) - pd.Timedelta(days=1)
             - pd.DateOffset(months=VAL_MONTHS))
dfn_f, sd_enc_f, sd_h_f = normalize_for_fold(df, fit_end_f)
imp_ds = GWLDataset(dfn_f, PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT, sd_enc_f, sd_h_f,
                    split_start=FOLDS[-1]['test_start'], split_end=FOLDS[-1]['test_end'], stride=5)

pw, fw, sw = [], [], []
with torch.no_grad():
    for i, b in enumerate(DataLoader(imp_ds, batch_size=EVAL_BATCH, shuffle=False)):
        if i >= 20: break
        sc, sx, px, fx = [t.to(DEVICE) for t in b[:4]]
        _, vsn, _ = model_final(sc, sx, px, fx)
        pw.append(vsn['past'].cpu().mean(dim=(0,1)))
        fw.append(vsn['future'].cpu().mean(dim=(0,1)))
        sw.append(vsn['static'].cpu().mean(dim=0))

static_labels = ['categorical(district+aquifer)'] + STATIC_CONT
fig, axes = plt.subplots(1, 3, figsize=(21, 10))
for ax, imp, names, color, title in [
    (axes[0], torch.stack(pw).mean(0).numpy(), PAST_CONT,     'steelblue', 'Past'),
    (axes[1], torch.stack(fw).mean(0).numpy(), FUTURE_CONT,   'coral',     'Known future'),
    (axes[2], torch.stack(sw).mean(0).numpy(), static_labels, 'seagreen',  'Static'),
]:
    idx = np.argsort(imp)[::-1]
    ax.barh([names[i] for i in idx], imp[idx], color=color)
    ax.set_title(f'{title} (VSN weight)'); ax.invert_yaxis(); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR/'vsn_importance.png', dpi=150); plt.show()

top = np.argsort(torch.stack(pw).mean(0).numpy())[::-1][:8]
print('Top past features:', [PAST_CONT[i] for i in top])
print('''
  Sanity check: if a climatology-filled feature (et_mm / soil_moisture on the old
  WRIS feed) ranks highly, the model is leaning on a seasonal template dressed up
  as a physical driver. Cross-check against the covariate reality table above.''')
del model_final, imp_ds, dfn_f; gc.collect()

In [ ]:
# ── Download everything ──────────────────────────────────────────────────────
import shutil
shutil.make_archive('tft_v3_outputs', 'zip', 'outputs')
files.download('tft_v3_outputs.zip')
print('Files in outputs/:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name:<45} {p.stat().st_size/1024:>8.1f} KB')

## What to take to the write-up

1. **The headline is t+14 and t+30 on the daily target, and all three horizons on the
   weekly-mean target.** Quote MASE and skill score together — *"reduces error by X% versus
   assuming the water table does not change"* — and attach the Diebold–Mariano p-value.

2. **The per-district table must be read next to the tier column.** A district where the TFT
   *and* the ridge both lose to persistence is a data-collection finding, not a modelling
   failure. Six of v2's seven failures are in that category.

3. **The data shortcomings are results in their own right.** They are the concrete argument
   for what to collect next:
   - Tier C districts have 2–4 wells and are missing 45–75% of 2023. Nashik's water table
     shows a **−0.15 correlation with rainfall** — it is not measuring a recharge-driven
     aquifer at all. More wells is the only fix.
   - Latur (tier D) is one well, 44% observed, with zero test windows in three of six folds.
   - ET and soil moisture had to be re-sourced because the original feed stopped publishing
     14 months before the last fold ended.

4. **Do not quote v2's "205,173 parameters".** That configuration builds 1,231,159. The
   number printed by this notebook is the real one.

5. **Reservoir storage** (`cwc_reservoir.py`, needs a free data.gov.in key) is still the main
   uncollected physical driver, and is the obvious next data item after more wells.
